# Étude de Résilience Eco-QNN
Ce notebook automatise la génération de données pour analyser :
1. L'influence de la profondeur du circuit sur la précision (1 qubit, bruité).
2. L'efficacité d'AdaBoost pour stabiliser les performances.

**Objectif :** Trouver le compromis optimal entre profondeur, bruit et agrégation.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split

# --- CONFIGURATION GLOBALE ---
N_DIM = 3
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42
NOISE_RATE = 0.02
MAXITER = 100
REUPLOAD_SHOTS = 1024

print("Configuration chargée.")

Configuration chargée.


In [6]:
def generate_nsphere_data(n_samples, n_dim, seed=None):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1, 1, (n_samples, n_dim))
    radius = np.sqrt(n_dim / 3)
    dist_sq = np.sum(X**2, axis=1)
    y = (dist_sq >= radius**2).astype(int)
    return X, y

def get_noisy_backend(rate):
    nm = NoiseModel()
    err1 = depolarizing_error(rate, 1)
    nm.add_all_qubit_quantum_error(err1, ['u', 'h', 'ry', 'rz'])
    err2 = depolarizing_error(rate * 5, 2)
    nm.add_all_qubit_quantum_error(err2, ['cx'])
    return AerSimulator(noise_model=nm, method="density_matrix")

def create_reupload_circuit(x, theta, omega, rc):
    qc = QuantumCircuit(1)
    for r in range(rc):
        qc.h(0)
        qc.u(theta[r,0] + omega[r,0]*x[0], 
             theta[r,1] + omega[r,1]*x[1], 
             theta[r,2] + omega[r,2]*x[2], 0)
    return qc

def get_probs_batch(circuits, backend, shots=REUPLOAD_SHOTS):
    measured_circuits = []
    for qc in circuits:
        qc_m = qc.copy()
        qc_m.measure_all()
        measured_circuits.append(qc_m)
    
    transpiled = transpile(measured_circuits, backend)
    result = backend.run(transpiled, shots=shots).result()
    
    probs = []
    for qc_t in transpiled:
        counts = result.get_counts(qc_t)
        p0 = counts.get('0', 0) / shots
        p1 = counts.get('1', 0) / shots
        probs.append((p0, p1))
    return probs

def cost_function(params, X, y, rc, backend, sample_weights=None):
    theta = params[:3*rc].reshape(rc, 3)
    omega = params[3*rc:].reshape(rc, 3)
    
    circuits = [create_reupload_circuit(x, theta, omega, rc) for x in X]
    probs = get_probs_batch(circuits, backend)
    
    if sample_weights is None:
        sample_weights = np.ones(len(y)) / len(y)
        
    total_cost = 0.0
    for i, y_target in enumerate(y):
        p0, p1 = probs[i]
        y_expected = (1.0, 0.0) if y_target == 0 else (0.0, 1.0)
        total_cost += sample_weights[i] * ((p0 - y_expected[0])**2 + (p1 - y_expected[1])**2)
    return 0.5 * total_cost / np.sum(sample_weights)

def train_learner(X, y, rc, backend, weights=None, seed=SEED):
    rng = np.random.default_rng(seed)
    init = rng.uniform(-np.pi, np.pi, size=6 * rc)
    res = minimize(cost_function, init, args=(X, y, rc, backend, weights), 
                   method='COBYLA', options={'maxiter': MAXITER})
    theta = res.x[:3*rc].reshape(rc, 3)
    omega = res.x[3*rc:].reshape(rc, 3)
    return theta, omega

def predict(X, theta, omega, rc, backend):
    circuits = [create_reupload_circuit(x, theta, omega, rc) for x in X]
    probs = get_probs_batch(circuits, backend)
    return np.array([0 if p0 >= p1 else 1 for p0, p1 in probs])

## 1. Simulation : Précision vs Profondeur
On fait varier le nombre de couches de 1 à 16.

In [7]:
X, y = generate_nsphere_data(N_SAMPLES, N_DIM, SEED)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
backend = get_noisy_backend(NOISE_RATE)

depths = [1, 2, 4, 6, 8, 10, 12, 14, 16]
depth_results = []

print("Début de la simulation Accuracy vs Profondeur...")
for d in depths:
    start = time.time()
    theta, omega = train_learner(X_train, y_train, d, backend)
    y_pred = predict(X_test, theta, omega, d, backend)
    acc = np.mean(y_pred == y_test)
    elapsed = time.time() - start
    depth_results.append({"depth": d, "accuracy": acc, "time": elapsed})
    print(f"Profondeur {d} : Accuracy = {acc:.4f} ({elapsed:.1f}s)")

df_depth = pd.DataFrame(depth_results)
df_depth.to_csv("RESULTS/study_depth_accuracy.csv", index=False)
print("Données sauvegardées dans RESULTS/study_depth_accuracy.csv")

Début de la simulation Accuracy vs Profondeur...
Profondeur 1 : Accuracy = 0.6850 (814.4s)
Profondeur 2 : Accuracy = 0.7400 (1046.4s)
Profondeur 4 : Accuracy = 0.7050 (1054.2s)


capi_return is NULL
Call-back cb_calcfc_in__cobyla__user__routines failed.


KeyboardInterrupt: 

## 2. Simulation : Précision vs Étapes AdaBoost
On fixe la profondeur à 8 couches et on augmente le nombre d'estimateurs.

In [8]:
def run_adaboost_study(X_tr, y_tr, X_te, y_te, rc, n_max, backend):
    n = len(y_tr)
    weights = np.ones(n) / n
    estimators = []
    boost_results = []
    
    for i in range(n_max):
        theta, omega = train_learner(X_tr, y_tr, rc, backend, weights, seed=SEED+i)
        y_pred_tr = predict(X_tr, theta, omega, rc, backend)
        
        incorrect = (y_pred_tr != y_tr).astype(float)
        eps = np.dot(weights, incorrect)
        eps = np.clip(eps, 1e-10, 1-1e-10)
        alpha = 0.5 * np.log((1 - eps) / eps)
        
        y_signed = np.where(y_tr == 0, -1.0, 1.0)
        h_signed = np.where(y_pred_tr == 0, -1.0, 1.0)
        weights *= np.exp(-alpha * y_signed * h_signed)
        weights /= np.sum(weights)
        
        estimators.append((theta, omega, alpha))
        
        # Evaluation de l'ensemble actuel
        scores_te = np.zeros(len(y_te))
        for t, o, a in estimators:
            h_te = np.where(predict(X_te, t, o, rc, backend) == 0, -1.0, 1.0)
            scores_te += a * h_te
        
        y_final = np.where(scores_te >= 0, 1, 0)
        acc = np.mean(y_final == y_te)
        boost_results.append({"n_estimators": i+1, "accuracy": acc})
        print(f"AdaBoost étape {i+1} : Accuracy = {acc:.4f}")
        
    return pd.DataFrame(boost_results)

FIXED_DEPTH = 8
N_BOOST_MAX = 10

print(f"Début de la simulation AdaBoost (Profondeur fixe={FIXED_DEPTH})...")
df_boost = run_adaboost_study(X_train, y_train, X_test, y_test, FIXED_DEPTH, N_BOOST_MAX, backend)
df_boost.to_csv("RESULTS/study_adaboost_accuracy.csv", index=False)
print("Données sauvegardées dans RESULTS/study_adaboost_accuracy.csv")

Début de la simulation AdaBoost (Profondeur fixe=8)...
AdaBoost étape 1 : Accuracy = 0.7350


## 3. Visualisation des résultats

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(df_depth['depth'], df_depth['accuracy'], 'o-', color='green')
plt.title("Précision vs Profondeur (1 qubit bruité)")
plt.xlabel("Nombre de couches (RC_REUPLOAD)")
plt.ylabel("Accuracy")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(df_boost['n_estimators'], df_boost['accuracy'], 's-', color='blue')
plt.title(f"Précision vs Étapes AdaBoost (Profondeur={FIXED_DEPTH})")
plt.xlabel("Nombre d'estimateurs")
plt.ylabel("Accuracy")
plt.grid(True)

plt.tight_layout()
plt.show()

: 